# Class Activity: Introduction to SQL 4-27

In [10]:
import sqlite3
import pandas as pd

### Connect and Create the Database
Example: Strudent Registration Records

In [11]:
conn = sqlite3.connect('university_admissions.db')
cursor = conn.cursor() # need this and the next line of code because we're doing this in python
cursor.execute('''
CREATE TABLE IF NOT EXISTS student_records (
    id INTEGER PRIMARY KEY, 
    student_name TEXT,
    exam_score INTEGER,
    status TEXT,
    fees_paid BOOLEAN
)
''') # name and define type of variable for each column 
# we have structure for student_records now, but it's empty

student_data = [
    (1, 'Alice', 94, 'Enrolled', 1),
    (2, 'Bob', 58, 'Enrolled', 1),
    (3, 'Charlie', 82, 'Waitlisted', 0),  
    (4, 'David', 45, 'Enrolled', 1),
    (5, 'Eve', 91, 'Waitlisted', 0)       
]

cursor.executemany('INSERT OR REPLACE INTO student_records VALUES (?, ?, ?, ?, ?)', student_data) # ? is a place holder for the values in the array
#since we had multiple records in student_data, we used executemany
conn.commit() #commit changes (i.e. finalize)


### Query the Data: Select, Filter etc.

In [12]:
query = '''
SELECT 
    student_name, 
    exam_score,
    CASE 
        WHEN exam_score >= 60 THEN 'PASS'
        ELSE 'FAIL - Must Retake'
    END AS exam_result
FROM student_records
WHERE status = 'Enrolled' AND fees_paid = 1
ORDER BY exam_score DESC
''' # END AS exam result creates new column; WHERE filters for specific conditions; ORDER arranges the data

print("--- Official Enrolled Student Records ---")
df = pd.read_sql_query(query, conn) #read_sql_query is better than execute for commands that give you output. 
#Execute is fine for commands that modify or create a table 
print(df)


print("\n--- Waitlisted Students---")
waitlist_df = pd.read_sql_query("SELECT student_name, exam_score FROM student_records WHERE status = 'Waitlisted'", conn)
# put everything in the read_sql_query argument
print(waitlist_df)


--- Official Enrolled Student Records ---
  student_name  exam_score         exam_result
0        Alice          94                PASS
1          Bob          58  FAIL - Must Retake
2        David          45  FAIL - Must Retake

--- Waitlisted Students---
  student_name  exam_score
0      Charlie          82
1          Eve          91


### Updating and Deleting

In [15]:
print("--- Updating Bob's Score ---")
cursor.execute("UPDATE student_records SET exam_score = 75 WHERE student_name = 'Bob'")
conn.commit() # change Bob's score to 75

print("--- Updated Student Records ---")
df = pd.read_sql_query("SELECT * FROM student_records", conn)
print(df) #view studnt record s now SELECT * means select alll

cursor.execute("DELETE FROM student_records WHERE student_name = 'Charlie'")
conn.commit() # make sure you have  a condition when you're dleteing3

print("-"*50)
print("--- Deleting Charlie (Dropped) ---")
df = pd.read_sql_query("SELECT * FROM student_records", conn)
print(df)



--- Updating Bob's Score ---
--- Updated Student Records ---
   id student_name  exam_score      status  fees_paid
0   1        Alice          94    Enrolled          1
1   2          Bob          75    Enrolled          1
2   3      Charlie          82  Waitlisted          0
3   4        David          45    Enrolled          1
4   5          Eve          91  Waitlisted          0
--------------------------------------------------
--- Deleting Charlie (Dropped) ---
   id student_name  exam_score      status  fees_paid
0   1        Alice          94    Enrolled          1
1   2          Bob          75    Enrolled          1
2   4        David          45    Enrolled          1
3   5          Eve          91  Waitlisted          0


In [24]:
#insert a row:
cursor.execute("INSERT INTO student_records VALUES (7, 'Frankie', 81, 'Enrolled', 1)")
conn.commit()

print("--- Updated Student Records ---")
df = pd.read_sql_query("SELECT * FROM student_records", conn)
print(df) #view studnt record s now SELECT * means select alll


--- Updated Student Records ---
   id student_name  exam_score      status  fees_paid
0   1        Alice          94    Enrolled          1
1   2          Bob          75    Enrolled          1
2   4        David          45    Enrolled          1
3   5          Eve          91  Waitlisted          0
4   6      Frankie          81    Enrolled          1
5   7      Frankie          81    Enrolled          1


In [35]:
#update Eve to Enrolled
print("--updating Eve's status ---")
cursor.execute("UPDATE student_records SET status = 'Enrolled' WHERE student_name = 'Eve'")
conn.commit()

print('--Updated Student Records ---')
df= pd.read_sql_query("SELECT * FROM student_records", conn)
print(df)

conn.close

--updating Eve's status ---


OperationalError: no such table: student_records

## Creating a new database: shopping cart

In [34]:
conn = sqlite3.connect("shopping.db")
cursor = conn.cursor()
cursor.execute('''
CREATE TABLE IF NOT EXISTS shopping_cart ( 
    product TEXT PRIMARY KEY,
    price FLOAT,
    quantity INTEGER,
    on_sale BOOLEAN
)''' 
)

customer_cart = [
    ( "drill", 69.99, 1, 0),
    ("paper_towels", 5.40, 1, 1),
    ("paint_scraper", 2.48, 2, 0),
    ("Gatorate 8pk", 6.99, 3, 0)
]

cursor.executemany('INSERT OR REPLACE INTO shopping_cart VALUES (?, ?, ?, ?)', customer_cart)
conn.close()

OperationalError: database is locked

In [32]:
]

AttributeError: 'sqlite3.Connection' object has no attribute 'end'